# ETL de dengue en Google Colab

El notebook obtiene los ZIP oficiales de 2020–2026 en memoria, procesa el CSV por bloques y guarda solo el Parquet semanal en Google Drive. No escribe las bases originales en tu computadora. La Secretaría de Salud indica que estas bases históricas se publican desde 2020; no se incluye 2015–2019. El clima ERA5-Land aún no está integrado.

In [ ]:
%pip install -q pandas pyarrow

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

from io import BytesIO
from pathlib import Path
from urllib.request import Request, urlopen
from zipfile import BadZipFile, ZipFile

import pandas as pd

OUTPUT_PATH = Path(
    '/content/drive/MyDrive/dengue_confirmed_weekly_2020_2026.parquet'
)
CHUNKSIZE = 50_000

RELEASES = (
    (2020, 'https://datosabiertos.salud.gob.mx/gobmx/salud/datos_abiertos/etv/historicos/2020/datos_abiertos_dengue_311220.zip'),
    (2021, 'https://datosabiertos.salud.gob.mx/gobmx/salud/datos_abiertos/etv/historicos/2021/datos_abiertos_dengue_301221.zip'),
    (2022, 'https://datosabiertos.salud.gob.mx/gobmx/salud/datos_abiertos/etv/historicos/2022/datos_abiertos_dengue_291222.zip'),
    (2023, 'https://datosabiertos.salud.gob.mx/gobmx/salud/datos_abiertos/etv/historicos/2023/datos_abiertos_dengue_271223.zip'),
    (2024, 'https://datosabiertos.salud.gob.mx/gobmx/salud/datos_abiertos/etv/historicos/2024/datos_abiertos_dengue_191224.zip'),
    (2025, 'https://datosabiertos.salud.gob.mx/gobmx/salud/datos_abiertos/etv/historicos/2025/datos_abiertos_dengue_111225.zip'),
    (2026, 'https://datosabiertos.salud.gob.mx/gobmx/salud/datos_abiertos/etv/historicos/2026/datos_abiertos_dengue_151026.zip'),
)


In [ ]:
weekly_chunks = []
raw_records = 0
confirmed_before_cleaning = 0
confirmed_after_cleaning = 0
symptom_after_update = 0

for release_year, url in RELEASES:
    request = Request(url, headers={'User-Agent': 'DengueDataEngineeringLab/1.0'})
    print(f'Leyendo corte oficial {release_year}...')
    with urlopen(request, timeout=60) as response:
        archive_bytes = response.read()

    try:
        with ZipFile(BytesIO(archive_bytes)) as archive:
            damaged_member = archive.testzip()
            if damaged_member is not None:
                raise BadZipFile(f'ZIP dañado en {release_year}: {damaged_member}')
            csv_members = [name for name in archive.namelist() if name.lower().endswith('.csv')]
            if len(csv_members) != 1:
                raise ValueError(f'Se esperaba un CSV en {release_year}; encontrados: {len(csv_members)}')

            with archive.open(csv_members[0]) as csv_file:
                for chunk in pd.read_csv(csv_file, chunksize=CHUNKSIZE, low_memory=False):
                    raw_records += len(chunk)
                    status = pd.to_numeric(chunk['ESTATUS_CASO'], errors='coerce')
                    confirmed_before_cleaning += int((status == 2).sum())
                    df = chunk.loc[status == 2].copy()
                    df['FECHA_SIGN_SINTOMAS'] = pd.to_datetime(
                        df['FECHA_SIGN_SINTOMAS'], format='mixed', dayfirst=True, errors='coerce'
                    )
                    df['ENTIDAD_RES'] = pd.to_numeric(df['ENTIDAD_RES'], errors='coerce')
                    df['MUNICIPIO_RES'] = pd.to_numeric(df['MUNICIPIO_RES'], errors='coerce')
                    df = df.dropna(
                        subset=['FECHA_SIGN_SINTOMAS', 'ENTIDAD_RES', 'MUNICIPIO_RES']
                    ).copy()

                    updates = pd.to_datetime(
                        df['FECHA_ACTUALIZACION'], format='mixed', dayfirst=True, errors='coerce'
                    )
                    symptom_after_update += int(
                        (df['FECHA_SIGN_SINTOMAS'] > updates).sum()
                    )
                    df['ENTIDAD_RES'] = df['ENTIDAD_RES'].astype('int64').astype(str).str.zfill(2)
                    df['MUNICIPIO_RES'] = df['MUNICIPIO_RES'].astype('int64').astype(str).str.zfill(3)
                    df['CVE_GEO'] = df['ENTIDAD_RES'] + df['MUNICIPIO_RES']
                    iso_dates = df['FECHA_SIGN_SINTOMAS'].dt.isocalendar()
                    df['EPI_YEAR'] = iso_dates['year'].astype('int32')
                    df['EPI_WEEK'] = iso_dates['week'].astype('int8')
                    confirmed_after_cleaning += len(df)
                    weekly_chunks.append(
                        df.groupby(
                            ['CVE_GEO', 'EPI_YEAR', 'EPI_WEEK'], as_index=False
                        ).size().rename(columns={'size': 'CONFIRMED_CASES'})
                    )
    finally:
        del archive_bytes

weekly = (
    pd.concat(weekly_chunks, ignore_index=True)
    .groupby(['CVE_GEO', 'EPI_YEAR', 'EPI_WEEK'], as_index=False)['CONFIRMED_CASES']
    .sum()
)
OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)
weekly.to_parquet(OUTPUT_PATH, index=False)

assert not weekly.duplicated(['CVE_GEO', 'EPI_YEAR', 'EPI_WEEK']).any()
assert weekly['CVE_GEO'].str.fullmatch(r'\d{5}').all()
assert weekly['EPI_WEEK'].between(1, 53).all()
assert int(weekly['CONFIRMED_CASES'].sum()) == confirmed_after_cleaning

print(f'Registros fuente: {raw_records:,}')
print(f'Casos confirmados válidos: {confirmed_after_cleaning:,} / {confirmed_before_cleaning:,}')
print(f'Filas municipio-semana: {len(weekly):,}')
if symptom_after_update:
    print(f'ADVERTENCIA: {symptom_after_update:,} casos tienen síntomas posteriores a FECHA_ACTUALIZACION; se conservaron para revisión.')
print(f'Parquet guardado en Drive: {OUTPUT_PATH}')

In [ ]:
weekly_check = pd.read_parquet(OUTPUT_PATH)
print(f'Filas: {len(weekly_check):,} | Casos: {weekly_check["CONFIRMED_CASES"].sum():,}')
display(weekly_check.head())